# Interactive editors

`mom6_forge` has two notebook widgets:

* `VGridCreator` builds a vertical grid (`VGrid`), uniform or hyperbolic, and keeps a library of saved grids.
* `TopoEditor` edits a bathymetry (`Topo`) cell by cell: depths, land and ocean, basins, with undo and redo.

Each widget is a front end over ordinary `mom6_forge` calls, so whatever you do in one can also be done in Python.

The widgets run in JupyterLab, Jupyter Notebook 7, VS Code, Colab and marimo; on NCAR's JupyterHub, install anywidget for your user once first. The [widgets page](../widgets.md) explains that, describes every control, and has live versions of both widgets that run in your browser.

On the documentation website the widgets below are shown as they were when the documentation was built. No Python runs there, so most controls do nothing; run the notebook to use them.

## 1. A small test domain

A 6° x 4° box at 0.1° (60 x 40 cells) with a bowl-shaped bathymetry, 200 m at the edges and 2000 m in the middle.

`git=True` keeps an edit history in a `TopoLibrary/` folder next to the notebook. That gives TopoEditor its undo, redo, reset, tag and branch controls. With `git=False` (the default) the editor still edits, but keeps no history.

In [1]:
from mom6_forge.grid import Grid
from mom6_forge.topo import Topo
from mom6_forge.vgrid import VGrid
from mom6_forge.widgets import TopoEditor, VGridCreator

grid = Grid(lenx=6.0, leny=4.0, resolution=0.1, xstart=283.0, ystart=35.0, name="editor_demo")
topo = Topo(grid, min_depth=10.0, git=True)
topo.set_bowl(max_depth=2000.0, dedge=200.0)

Wall off the south-east corner with land, so that it becomes a separate basin to try the basin tools on. This uses `topo.apply_edits`, the same call TopoEditor makes for each edit. Cells are `(j, i)` index pairs.

In [2]:
wall = [(j, 45) for j in range(16)] + [(15, i) for i in range(45, 60)]
changed = topo.apply_edits([{"op": "land", "cells": wall}])

The manual mask is not initialized. Initializing it now from the depth raw mask


## 2. VGridCreator

Start from a hyperbolic grid. Passing `topo` lets the widget warn when the grid is shallower than the deepest point of the bathymetry; here it is (1500 m against 2000 m).

* **Uniform / Hyperbolic**: uniform layers all have the same thickness; hyperbolic layers change thickness smoothly with depth.
* **Levels** and **Depth (m)**: the number of layers and the total depth.
* **Bottom/top thickness ratio** (hyperbolic only): bottom layer thickness divided by top layer thickness. Above 1 gives thin layers near the surface and thick ones at depth.
* **Reset to the starting grid**: back to the grid the widget was opened with.
* **Library**: **Save** writes `VGridLibrary/vgrid_<name>.nc` with your message, **Load** reads a saved grid back. Saving under an existing name asks before replacing it.

The plot shows each layer as a bar as wide as its thickness, at its depth.

In [3]:
vgrid = VGrid.hyperbolic(nk=20, depth=1500.0, ratio=8.0, name="editor_demo")
vgc = VGridCreator(vgrid=vgrid, topo=topo)
vgc

The grid in the widget is `vgc.vgrid`, an ordinary `VGrid`:

In [4]:
vgrid = vgc.vgrid

## 3. TopoEditor

The map is drawn in grid-index space: one pixel per cell, `i` to the right and `j` up. That keeps picking a cell exact on rotated, curvilinear and polar grids. Hover over a cell to see its `i`, `j`, longitude, latitude and depth.

Things to try:

1. Switch the view to **basinmask**. The walled-off corner is its own basin.
2. With the **Select** tool, click a cell in the large basin, then **Erase disconnected basins**. The corner becomes land.
3. Click **Undo**, and the corner comes back.
4. Drag a box, type a depth and click **Set depth**. Or pick **Paint land** and paint over a few cells.

The editor changes `topo` in place.

In [5]:
editor = TopoEditor(topo)
editor

## 4. The same edits in Python

Every editor action is one `topo.apply_edits` entry, so a script or an AI assistant can make the same edits. The ops are `depth`, `land`, `ocean`, `clear_mask`, `min_depth`, `fill_from_stats`, `erase_selected_basin` and `erase_disconnected_basin`; the `Topo.apply_edits` docstring gives the fields of each. Each entry is one history step, and `topo.undo()`, `topo.redo()` and `topo.reset()` step through the history (they need `git=True`).

An open editor does not redraw for edits made from Python; run a cell with just `editor` to show it again from the current `topo`. The cell below changes nothing in the end, so the editor above stays current.

In [6]:
# Deepen two cells, then take it back.
changed = topo.apply_edits([{"op": "depth", "cells": [(20, 10), (20, 11)], "value": 2500.0}])
changed = topo.undo()

## 5. Save the results

In [7]:
topo.write_topo_file("ocean_topog_editor_demo.nc")
vgrid.write_vgrid_file("ocean_vgrid_editor_demo.nc");